In [16]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
invoices = pd.read_csv(BASE_PATH + 'invoices.csv')
payments = pd.read_csv(BASE_PATH + 'payments.csv')

print("Invoices:", invoices.shape)
print("Payments:", payments.shape)

Invoices: (18033, 10)
Payments: (19257, 5)


In [18]:
# Find payments whose invoice_id has no match in invoices at all
orphaned_payments = payments[~payments['invoice_id'].isin(invoices['invoice_id'])]

print("Total payments:", len(payments))
print("Orphaned payments (no matching invoice_id):", len(orphaned_payments))
print("Percentage orphaned:", len(orphaned_payments) / len(payments) * 100)

Total payments: 19257
Orphaned payments (no matching invoice_id): 0
Percentage orphaned: 0.0


In [19]:
# --- Identify affected invoice_ids (duplicated in invoices.csv) ---
invoice_id_counts = invoices['invoice_id'].value_counts()
affected_invoice_ids = invoice_id_counts[invoice_id_counts > 1].index

# --- Identify affected payment_ids (duplicated in payments.csv) ---
payment_id_counts = payments['payment_id'].value_counts()
affected_payment_ids = payment_id_counts[payment_id_counts > 1].index

print(f"Affected invoice_ids: {len(affected_invoice_ids)}")
print(f"Affected payment_ids: {len(affected_payment_ids)}")

Affected invoice_ids: 196
Affected payment_ids: 201


In [20]:
excluded_payments = payments[payments['payment_id'].isin(affected_payment_ids)]

In [21]:
invoice_id_counts = invoices['invoice_id'].value_counts()
affected_invoice_ids = invoice_id_counts[invoice_id_counts > 1].index

payment_id_counts = payments['payment_id'].value_counts()
affected_payment_ids = payment_id_counts[payment_id_counts > 1].index

excluded_invoices = invoices[invoices['invoice_id'].isin(affected_invoice_ids)].copy()
excluded_payments = payments[payments['payment_id'].isin(affected_payment_ids)].copy()

print(len(affected_invoice_ids), len(excluded_invoices))
print(len(affected_payment_ids), len(excluded_payments))

196 393
201 403


In [22]:
print("=== Invoice amount distribution (grand_total) ===")
print("Full table describe:\n", invoices['grand_total'].describe())
print("Excluded describe:\n", excluded_invoices['grand_total'].describe())

print("\n=== Payment amount distribution ===")
print("Full table describe:\n", payments['payment_amount'].describe())
print("Excluded describe:\n", excluded_payments['payment_amount'].describe())

=== Invoice amount distribution (grand_total) ===
Full table describe:
 count    1.803300e+04
mean     1.622248e+06
std      1.374170e+06
min      3.658000e+02
25%      5.405440e+05
50%      1.287381e+06
75%      2.348676e+06
max      1.045198e+07
Name: grand_total, dtype: float64
Excluded describe:
 count    3.930000e+02
mean     1.665026e+06
std      1.340048e+06
min      3.658000e+02
25%      6.006450e+05
50%      1.372629e+06
75%      2.372543e+06
max      6.930637e+06
Name: grand_total, dtype: float64

=== Payment amount distribution ===
Full table describe:
 count    1.925700e+04
mean     1.210411e+06
std      1.234690e+06
min      1.913000e+01
25%      2.692513e+05
50%      8.207170e+05
75%      1.769986e+06
max      9.970301e+06
Name: payment_amount, dtype: float64
Excluded describe:
 count    4.030000e+02
mean     1.267410e+06
std      1.281764e+06
min      2.689500e+02
25%      2.711453e+05
50%      8.699967e+05
75%      1.832847e+06
max      7.315783e+06
Name: payment_amount

In [23]:
print(invoices.columns.tolist())
print(payments.columns.tolist())

['invoice_id', 'so_id', 'customer_id', 'branch_id', 'invoice_date', 'due_date', 'total_order_value', 'total_gst_amount', 'grand_total', 'payment_status']
['payment_id', 'invoice_id', 'payment_date', 'payment_amount', 'payment_method']


In [24]:
# Step 1: build clean tables
invoices_clean = invoices[~invoices['invoice_id'].isin(affected_invoice_ids)].copy()
payments_clean = payments[~payments['payment_id'].isin(affected_payment_ids)].copy()

print("invoices_clean:", len(invoices_clean), "(expect 17640)")
print("payments_clean:", len(payments_clean), "(expect 18854)")

# Step 2: how many clean payments point to an excluded invoice? (these will drop in the merge)
pay_on_excluded_inv = payments_clean['invoice_id'].isin(affected_invoice_ids).sum()
print("clean payments pointing to an excluded invoice:", pay_on_excluded_inv)

# Step 3: merge (many payments -> one invoice), enforced by validate
merged = payments_clean.merge(
    invoices_clean[['invoice_id', 'invoice_date', 'due_date', 'grand_total']],
    on='invoice_id',
    how='inner',
    validate='m:1'
)

print("merged rows:", len(merged))
print("reconciliation:", len(payments_clean), "-", pay_on_excluded_inv, "=", len(payments_clean) - pay_on_excluded_inv)
print("dtypes:")
print(merged[['payment_date', 'due_date']].dtypes)

invoices_clean: 17640 (expect 17640)
payments_clean: 18854 (expect 18854)
clean payments pointing to an excluded invoice: 409
merged rows: 18445
reconciliation: 18854 - 409 = 18445
dtypes:
payment_date    object
due_date        object
dtype: object


In [25]:
merged['payment_date'] = pd.to_datetime(merged['payment_date'], errors='coerce')
merged['due_date'] = pd.to_datetime(merged['due_date'], errors='coerce')

print(merged[['payment_date', 'due_date']].dtypes)
print("Nulls after conversion:")
print(merged[['payment_date', 'due_date']].isna().sum())
print("Date ranges:")
print(merged['payment_date'].min(), "to", merged['payment_date'].max())
print(merged['due_date'].min(), "to", merged['due_date'].max())

payment_date    datetime64[ns]
due_date        datetime64[ns]
dtype: object
Nulls after conversion:
payment_date    0
due_date        0
dtype: int64
Date ranges:
2019-01-09 00:00:00 to 2025-03-20 00:00:00
2019-01-20 00:00:00 to 2025-03-14 00:00:00


In [26]:
merged['is_late'] = merged['payment_date'] > merged['due_date']
merged['days_late'] = (merged['payment_date'] - merged['due_date']).dt.days

late_count = merged['is_late'].sum()
total = len(merged)

print("Late payments:", late_count)
print("Total payments:", total)
print("Late Payment Rate:", late_count / total)
print()
print(merged['days_late'].describe())

Late payments: 7655
Total payments: 18445
Late Payment Rate: 0.4150176199512063

count    18445.000000
mean        -4.193874
std         25.219086
min        -60.000000
25%        -22.000000
50%         -5.000000
75%         14.000000
max         75.000000
Name: days_late, dtype: float64


In [27]:
late = merged[merged['is_late']]
on_time = merged[~merged['is_late']]

print("Late:", len(late))
print("On time or early:", len(on_time))
print("Sum:", len(late) + len(on_time))
print("Exactly on due date (0 days):", (merged['days_late'] == 0).sum())
print()
print(late['days_late'].describe())

Late: 7655
On time or early: 10790
Sum: 18445
Exactly on due date (0 days): 271

count    7655.000000
mean       20.018811
std        14.362825
min         1.000000
25%         9.000000
50%        17.000000
75%        28.000000
max        75.000000
Name: days_late, dtype: float64


In [28]:
print("is_late nulls:", merged['is_late'].isna().sum())
print()

# Headline: Late Payment Rate, three independent ways
r1 = merged['is_late'].mean()
r2 = merged.query('payment_date > due_date').shape[0] / merged.shape[0]
r3 = (merged['days_late'] > 0).sum() / len(merged)
print("Rate method 1:", r1)
print("Rate method 2:", r2)
print("Rate method 3:", r3)
print()

# Supporting: mean days late, three independent ways
m1 = merged.loc[merged['is_late'], 'days_late'].mean()
m2 = (merged.loc[merged['is_late'], 'payment_date'] - merged.loc[merged['is_late'], 'due_date']).dt.days.mean()
m3 = merged.loc[merged['days_late'] > 0, 'days_late'].sum() / (merged['days_late'] > 0).sum()
print("Mean days late method 1:", m1)
print("Mean days late method 2:", m2)
print("Mean days late method 3:", m3)
print()

# Supporting: median days late
print("Median days late:", late['days_late'].median())

is_late nulls: 0

Rate method 1: 0.4150176199512063
Rate method 2: 0.4150176199512063
Rate method 3: 0.4150176199512063

Mean days late method 1: 20.018811234487263
Mean days late method 2: 20.018811234487263
Mean days late method 3: 20.018811234487263

Median days late: 17.0


In [29]:
# Bring in branch and customer (safe: invoice_id is unique in invoices_clean)
inv_lookup = invoices_clean.set_index('invoice_id')
merged['branch_id'] = merged['invoice_id'].map(inv_lookup['branch_id'])
merged['customer_id'] = merged['invoice_id'].map(inv_lookup['customer_id'])
merged['due_year'] = merged['due_date'].dt.year

print("Nulls after mapping:", merged[['branch_id', 'customer_id']].isna().sum().to_dict())
print()

# Late rate AND median days late (among late payments) by branch
print("BY BRANCH")
by_branch = merged.groupby('branch_id').agg(
    payments=('is_late', 'size'),
    late_rate=('is_late', 'mean')
)
by_branch['median_days_late'] = late.groupby(merged['branch_id']).median(numeric_only=True)['days_late']
print(by_branch.round(4))
print()

# By due year
print("BY DUE YEAR")
by_year = merged.groupby('due_year').agg(
    payments=('is_late', 'size'),
    late_rate=('is_late', 'mean')
)
by_year['median_days_late'] = late.groupby(merged['due_year']).median(numeric_only=True)['days_late']
print(by_year.round(4))
print()

# Customer concentration
cust = merged['customer_id'].value_counts(normalize=True)
print("Number of customers:", merged['customer_id'].nunique())
print("Top 5 customers' share of payments:")
print(cust.head(5).round(4))

Nulls after mapping: {'branch_id': 0, 'customer_id': 0}

BY BRANCH
           payments  late_rate  median_days_late
branch_id                                       
AHM001         3196     0.3961              16.0
CHN001         3732     0.4861              19.0
DEL001         2723     0.3772              17.0
HYD001         2576     0.3820              17.0
KOL001         3488     0.3974              17.0
PUN001         2730     0.4315              16.0

BY DUE YEAR
          payments  late_rate  median_days_late
due_year                                       
2019          2717     0.4314              18.0
2020          3060     0.4170              18.0
2021          3089     0.4024              17.0
2022          3102     0.4281              17.0
2023          3101     0.4112              17.0
2024          3005     0.4153              17.0
2025           371     0.3046              18.0

Number of customers: 500
Top 5 customers' share of payments:
customer_id
C0475    0.0032
C0304 

In [30]:
y25 = merged[merged['due_year'] == 2025]
print("2025 payments by due month:")
print(y25.groupby(y25['due_date'].dt.month).agg(
    payments=('is_late', 'size'), late_rate=('is_late', 'mean')).round(4))
print()

ex25 = merged[merged['due_year'] != 2025]
print("Rate excluding 2025:", ex25['is_late'].mean(), "| rows:", len(ex25))
print()

inv25 = invoices_clean[pd.to_datetime(invoices_clean['due_date']).dt.year == 2025]
print("Invoices due in 2025:", len(inv25))
print(inv25['payment_status'].value_counts())

2025 payments by due month:
          payments  late_rate
due_date                     
1              270     0.3556
2               88     0.1818
3               13     0.0769

Rate excluding 2025: 0.41728449706761095 | rows: 18074

Invoices due in 2025: 346
payment_status
Paid              248
Partially Paid     64
Unpaid             34
Name: count, dtype: int64


In [31]:
samples = pd.concat([
    merged[merged['is_late']].sample(3, random_state=42),
    merged[merged['days_late'] < 0].sample(3, random_state=42),
    merged[merged['days_late'] == 0].sample(1, random_state=42)
])

for _, r in samples.iterrows():
    raw_pay = payments[payments['payment_id'] == r['payment_id']].iloc[0]
    raw_inv = invoices[invoices['invoice_id'] == r['invoice_id']].iloc[0]
    print("payment_id:", r['payment_id'], "| invoice_id:", r['invoice_id'])
    print("  RAW payment_date:", raw_pay['payment_date'], "| RAW due_date:", raw_inv['due_date'])
    print("  merged days_late:", r['days_late'], "| is_late:", r['is_late'])
    print()

payment_id: PAY-324335 | invoice_id: INV-655489
  RAW payment_date: 2022-05-12 | RAW due_date: 2022-05-09
  merged days_late: 3 | is_late: True

payment_id: PAY-920424 | invoice_id: INV-839270
  RAW payment_date: 2021-12-11 | RAW due_date: 2021-11-22
  merged days_late: 19 | is_late: True

payment_id: PAY-706069 | invoice_id: INV-734072
  RAW payment_date: 2019-10-22 | RAW due_date: 2019-10-02
  merged days_late: 20 | is_late: True

payment_id: PAY-548776 | invoice_id: INV-810413
  RAW payment_date: 2020-09-25 | RAW due_date: 2020-10-03
  merged days_late: -8 | is_late: False

payment_id: PAY-967482 | invoice_id: INV-524304
  RAW payment_date: 2022-10-27 | RAW due_date: 2022-10-29
  merged days_late: -2 | is_late: False

payment_id: PAY-949408 | invoice_id: INV-901717
  RAW payment_date: 2020-03-12 | RAW due_date: 2020-04-09
  merged days_late: -28 | is_late: False

payment_id: PAY-352541 | invoice_id: INV-451802
  RAW payment_date: 2024-05-02 | RAW due_date: 2024-05-02
  merged days_l

In [32]:
# Bring invoice_date into merged and make it a real date
invoices_clean['invoice_date'] = pd.to_datetime(invoices_clean['invoice_date'])
merged['invoice_date'] = merged['invoice_id'].map(
    invoices_clean.set_index('invoice_id')['invoice_date']
)

# 1) How many payments happened BEFORE their invoice?
pay_before_inv = merged[merged['payment_date'] < merged['invoice_date']]
print("Payment before invoice:", len(pay_before_inv))

# 2) How many invoices have a due date BEFORE the invoice date?
print("Due before invoice:", (merged['due_date'] < merged['invoice_date']).sum())

# 3) Late rate if we remove those payments
ok = merged[merged['payment_date'] >= merged['invoice_date']]
print("Rows left:", len(ok))
print("Late rate without them:", ok['is_late'].mean())

Payment before invoice: 0
Due before invoice: 0
Rows left: 18445
Late rate without them: 0.4150176199512063


In [33]:
# Raw join: no exclusions, like Week 1
raw = payments.merge(invoices[['invoice_id', 'invoice_date']], on='invoice_id', how='inner')
raw['payment_date'] = pd.to_datetime(raw['payment_date'])
raw['invoice_date'] = pd.to_datetime(raw['invoice_date'])

bad = raw[raw['payment_date'] < raw['invoice_date']]
print("Raw join rows:", len(raw))
print("Payment before invoice (raw):", len(bad))
print("...of these, invoice_id is a duplicate ID:", bad['invoice_id'].isin(affected_invoice_ids).sum())
print("...of these, payment_id is a duplicate ID:", bad['payment_id'].isin(affected_payment_ids).sum())

Raw join rows: 19678
Payment before invoice (raw): 208
...of these, invoice_id is a duplicate ID: 208
...of these, payment_id is a duplicate ID: 3
